# Naive Bayes 

We'll use sklearn.naive_bayes to train a spam classifier! Most of the code is just loading our training data into a pandas DataFrame that we can play with:

In [1]:
import os
import io
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

def readFiles(path):
    """
    Generator that yields (filename, message_body) pairs.
    Reads files in the given directory recursively.
    """
    for root, _, filenames in os.walk(path):
        for filename in filenames:
            file_path = os.path.join(root, filename)
            inBody = False
            lines = []
            with io.open(file_path, 'r', encoding='latin1') as f:
                for line in f:
                    if inBody:
                        lines.append(line)
                    elif line == '\n':
                        inBody = True
            message = '\n'.join(lines)
            yield file_path, message


def dataFrameFromDirectory(path, classification):
    """
    Build a DataFrame with all emails in a directory.
    Each row contains the email text and its class label.
    """
    rows = []
    index = []
    for filename, message in readFiles(path):
        rows.append({'message': message, 'class': classification})
        index.append(filename)
    return pd.DataFrame(rows, index=index)


# --- Build dataset ---
spam_df = dataFrameFromDirectory('emails/spam', 'spam')
ham_df  = dataFrameFromDirectory('emails/ham', 'ham')

# use pd.concat instead of deprecated DataFrame.append
data = pd.concat([spam_df, ham_df], ignore_index=True)

print("Dataset shape:", data.shape)
print(data.head())


Dataset shape: (3000, 2)
                                             message class
0  Dear Sir or Madam:\n\n\n\nPlease reply to \n\n...  spam
1  We guarantee you signups before you ever pay\n...  spam
2  Protect your financial well-being.\n\nPurchase...  spam
3  I thought you might like these:\n\n1) Slim Dow...  spam
4  <html><head></head><body bgcolor=3Dblack>\n\n<...  spam


Let's have a look at that DataFrame:

In [2]:
data.head()

,message,class
0,Dear Sir or Madam:\n\n\n\nPlease reply to \n\n...,spam
1,We guarantee you signups before you ever pay\n...,spam
2,Protect your financial well-being.\n\nPurchase...,spam
3,I thought you might like these:\n\n1) Slim Dow...,spam
4,<html><head></head><body bgcolor=3Dblack>\n\n<...,spam


In the slides we've seen the example of a Gaussian Naive Bayes Classifier, if this example we'll use a Multinomial Naive Bayes classifier. Explain it works? (explication ci-dessous)

Now we will use a CountVectorizer to split up each message into its list of words, and throw that into a MultinomialNB classifier. Call fit() and we've got a trained spam filter ready to go! It's just that easy.

In [3]:
vectorizer = CountVectorizer()
counts = vectorizer.fit_transform(data['message'].values)

classifier = MultinomialNB()
targets = data['class'].values
classifier.fit(counts, targets)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[2500., 500.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.18,-1.79]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U4](2,)","['ham','spam']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 62510)","[[832.,316., 51.,..., 0., 0., 0.], [297.,630., 21.,..., 1., 1., 1.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 62510)","[[ -6.57, -7.53, -9.34,...,-13.29,-13.29,-13.29], [ -7.13, -6.38, -9.74,...,-12.14,-12.14,-12.14]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,62510


Let's try it out:

In [4]:
examples = ['Free insurance plan!!', "Hi Bob, how about a game of golf tomorrow?"]
example_counts = vectorizer.transform(examples)
predictions = classifier.predict(example_counts)
print(predictions)

['spam' 'ham']


## Your Assignment

### Comment fonctionne le Naive Bayes multinomial ? (réponse à la question du cours)

**Principe.** On cherche la classe `c` (spam / ham) la plus probable sachant le message `d`. Par Bayes :

`P(c | d) ∝ P(c) · P(d | c)`

**Hypothèse « naïve ».** Les mots du message sont supposés **conditionnellement indépendants** sachant la classe, donc :

`P(d | c) = Π_i P(w_i | c)^{n_i}`

où `n_i` est le nombre d'occurrences du mot `w_i` dans le message (c'est le « sac de mots » produit par `CountVectorizer`).

**Modèle multinomial.** Pour chaque classe, les mots sont tirés d'une loi multinomiale de paramètres `θ_{c,i} = P(w_i | c)`. L'apprentissage consiste simplement à compter :

`θ_{c,i} = (N_{c,i} + α) / (N_c + α·|V|)`

- `N_{c,i}` : nombre total d'occurrences du mot `i` dans les messages de la classe `c`
- `N_c` : nombre total de mots de la classe `c`, `|V|` : taille du vocabulaire
- `α` (=1 par défaut dans sklearn) : **lissage de Laplace**, qui évite qu'un mot jamais vu dans une classe donne une probabilité nulle et annule tout le produit.

**Prédiction.** Pour éviter l'underflow, on travaille en logarithmes :

`ĉ = argmax_c [ log P(c) + Σ_i n_i · log θ_{c,i} ]`

**Différence avec le Naive Bayes gaussien vu en cours :** le gaussien modélise des variables *continues* par `P(x_j | c) = N(μ_{c,j}, σ²_{c,j})` ; le multinomial modélise des *comptages* (mots), d'où son usage naturel en classification de texte.

## Partie 1 — Tester le classifieur et l'évaluer

### 1.a Quelques emails de test supplémentaires

In [5]:
tests = [
    "Congratulations! You won a FREE lottery prize, click here to claim your money now!!!",
    "Cheap viagra and cialis online, no prescription needed, limited offer",
    "Make $$$ from home, guaranteed income, act now",
    "Hi Alice, the meeting is moved to 3pm tomorrow, please bring the project report.",
    "Can you review my python code before the deadline on Friday?",
    "Dear colleague, attached are the minutes of last week's seminar.",
]
pred = classifier.predict(vectorizer.transform(tests))
proba = classifier.predict_proba(vectorizer.transform(tests))
for t, p, pr in zip(tests, pred, proba):
    print(f"[{p:4s}] (P(spam)={pr[list(classifier.classes_).index('spam')]:.3f})  {t[:70]}")

[spam] (P(spam)=0.955)  Congratulations! You won a FREE lottery prize, click here to claim you
[spam] (P(spam)=0.968)  Cheap viagra and cialis online, no prescription needed, limited offer
[spam] (P(spam)=0.849)  Make $$$ from home, guaranteed income, act now
[ham ] (P(spam)=0.001)  Hi Alice, the meeting is moved to 3pm tomorrow, please bring the proje
[ham ] (P(spam)=0.000)  Can you review my python code before the deadline on Friday?
[ham ] (P(spam)=0.009)  Dear colleague, attached are the minutes of last week's seminar.


Les résultats sont conformes à l'intuition : les messages typés spam (loterie, pharmacie en ligne, « gagner de l'argent ») sont classés spam, les messages professionnels sont classés ham. Le message « Make $$$ from home… » est le moins sûr (P(spam) ≈ 0.85) : le classifieur ne connaît que le vocabulaire de son corpus.

### 1.b Découpage train / test

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    data['message'].values, data['class'].values,
    test_size=0.25, random_state=42, stratify=data['class'].values)

vec = CountVectorizer()
Xtr = vec.fit_transform(X_train)      # le vocabulaire est appris sur le train uniquement
Xte = vec.transform(X_test)

clf = MultinomialNB().fit(Xtr, y_train)
y_pred = clf.predict(Xte)

print("Taille train / test :", len(X_train), "/", len(X_test))
print("Accuracy test :", round(accuracy_score(y_test, y_pred), 4))
print("\nMatrice de confusion (lignes = réel, colonnes = prédit) [ham, spam] :")
print(confusion_matrix(y_test, y_pred, labels=['ham', 'spam']))
print("\n", classification_report(y_test, y_pred, digits=3))

Taille train / test : 2250 / 750
Accuracy test : 0.9573

Matrice de confusion (lignes = réel, colonnes = prédit) [ham, spam] :
[[625   0]
 [ 32  93]]

               precision    recall  f1-score   support

         ham      0.951     1.000     0.975       625
        spam      1.000     0.744     0.853       125

    accuracy                          0.957       750
   macro avg      0.976     0.872     0.914       750
weighted avg      0.959     0.957     0.955       750



### 1.c Validation croisée (5 plis)

Le `CountVectorizer` est placé **dans un `Pipeline`** pour être réappris sur chaque pli d'entraînement : on évite ainsi toute fuite d'information du pli de test vers le vocabulaire.

In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

pipe = make_pipeline(CountVectorizer(), MultinomialNB())
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, data['message'].values, data['class'].values, cv=cv, scoring='accuracy')

print("Accuracy par pli :", scores.round(4))
print(f"Accuracy moyenne : {scores.mean():.4f}  (± {scores.std():.4f})")

Accuracy par pli : [0.9517 0.9333 0.95   0.9683 0.9583]
Accuracy moyenne : 0.9523  (± 0.0115)


## Partie 2 — Naive Bayes **from scratch** (Breast Cancer Wisconsin)

Les 30 variables de ce jeu de données sont **continues** : on implémente donc un **Naive Bayes gaussien** à la main (NumPy uniquement), comme dans les slides :

- *Apprentissage* : pour chaque classe `c`, prior `P(c) = n_c / n`, et pour chaque variable `j` la moyenne `μ_{c,j}` et la variance `σ²_{c,j}`.
- *Prédiction* : `ĉ = argmax_c [ log P(c) + Σ_j log N(x_j ; μ_{c,j}, σ²_{c,j}) ]`, avec  
  `log N(x; μ, σ²) = -½ log(2πσ²) - (x-μ)² / (2σ²)`.
- Un petit terme `ε` est ajouté aux variances pour la stabilité numérique.

In [8]:
from sklearn import datasets

cancer = datasets.load_breast_cancer()
X, y = cancer.data, cancer.target
print("X :", X.shape, "| classes :", dict(zip(cancer.target_names, np.bincount(y))))

X : (569, 30) | classes : {np.str_('malignant'): np.int64(212), np.str_('benign'): np.int64(357)}


In [9]:
class MyGaussianNB:
    """Naive Bayes gaussien écrit à la main."""

    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y)
        self.classes_ = np.unique(y)
        self.priors_, self.means_, self.vars_ = {}, {}, {}
        eps = self.var_smoothing * X.var(axis=0).max()
        for c in self.classes_:
            Xc = X[y == c]
            self.priors_[c] = len(Xc) / len(X)          # P(c)
            self.means_[c]  = Xc.mean(axis=0)            # mu_{c,j}
            self.vars_[c]   = Xc.var(axis=0) + eps       # sigma^2_{c,j}
        return self

    def _log_posterior(self, X):
        X = np.asarray(X, float)
        out = np.zeros((X.shape[0], len(self.classes_)))
        for k, c in enumerate(self.classes_):
            mu, var = self.means_[c], self.vars_[c]
            log_lik = -0.5 * np.log(2 * np.pi * var) - (X - mu) ** 2 / (2 * var)
            out[:, k] = np.log(self.priors_[c]) + log_lik.sum(axis=1)
        return out

    def predict(self, X):
        return self.classes_[np.argmax(self._log_posterior(X), axis=1)]

    def predict_proba(self, X):
        lp = self._log_posterior(X)
        lp -= lp.max(axis=1, keepdims=True)              # stabilité (log-sum-exp)
        p = np.exp(lp)
        return p / p.sum(axis=1, keepdims=True)

    def score(self, X, y):
        return np.mean(self.predict(X) == np.asarray(y))

In [10]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

mynb = MyGaussianNB().fit(Xtr, ytr)
yp = mynb.predict(Xte)

print("Accuracy (mon NB, test) :", round(mynb.score(Xte, yte), 4))
print("\nMatrice de confusion (lignes = réel, colonnes = prédit) [malignant, benign] :")
print(confusion_matrix(yte, yp))
print("\n", classification_report(yte, yp, target_names=cancer.target_names, digits=3))

Accuracy (mon NB, test) : 0.9371

Matrice de confusion (lignes = réel, colonnes = prédit) [malignant, benign] :
[[46  7]
 [ 2 88]]

               precision    recall  f1-score   support

   malignant      0.958     0.868     0.911        53
      benign      0.926     0.978     0.951        90

    accuracy                          0.937       143
   macro avg      0.942     0.923     0.931       143
weighted avg      0.938     0.937     0.936       143



### Comparaison avec `sklearn.naive_bayes.GaussianNB` et validation croisée

In [11]:
from sklearn.naive_bayes import GaussianNB

skl = GaussianNB().fit(Xtr, ytr)
print("Accuracy sklearn GaussianNB :", round(skl.score(Xte, yte), 4))
print("Prédictions identiques à sklearn :", np.mean(skl.predict(Xte) == yp) * 100, "%")
print("Écart max des probabilités :", np.abs(skl.predict_proba(Xte) - mynb.predict_proba(Xte)).max())

# Validation croisée 5 plis, avec ma propre implémentation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
my_scores, sk_scores = [], []
for tr, te in cv.split(X, y):
    my_scores.append(MyGaussianNB().fit(X[tr], y[tr]).score(X[te], y[te]))
    sk_scores.append(GaussianNB().fit(X[tr], y[tr]).score(X[te], y[te]))
print("\nCV mon NB   :", np.round(my_scores, 4), f"-> moyenne {np.mean(my_scores):.4f}")
print("CV sklearn  :", np.round(sk_scores, 4), f"-> moyenne {np.mean(sk_scores):.4f}")

Accuracy sklearn GaussianNB : 0.9371
Prédictions identiques à sklearn : 100.0 %
Écart max des probabilités : 8.881784197001252e-16

CV mon NB   : [0.9649 0.9035 0.9298 0.9298 0.9646] -> moyenne 0.9385
CV sklearn  : [0.9649 0.9035 0.9298 0.9298 0.9646] -> moyenne 0.9385


### Conclusion

- **Spam (MultinomialNB)** : ≈ 95–96 % d'accuracy en train/test comme en validation croisée (5 plis, ≈ 95.2 % ± 1.2). Sur le split utilisé, aucun ham n'est classé spam (précision spam = 1.0) mais environ 26 % des spams passent pour du ham (rappel spam ≈ 0.74) : le modèle est prudent, ce qui est souhaitable pour un filtre anti-spam, et s'améliorerait avec plus de données.
- **Breast cancer (NB from scratch)** : notre implémentation gaussienne donne les **mêmes prédictions** que `GaussianNB` de scikit-learn, avec une précision d'environ 93–94 %. L'hypothèse d'indépendance des variables est fausse ici (beaucoup de mesures sont corrélées : rayon, périmètre, aire…), mais le Naive Bayes reste un bon classifieur de base car il n'a besoin que de la classe la plus probable, pas de probabilités parfaitement calibrées.